# Eigenvalue extraction tests in a single-phase RLC circuit in DP & EMT domains

In [67]:
import dpsimpy
import math
import villas.dataprocessing.readtools as rt

In [68]:
# function to assert that two complex numbers are close
def assert_complex_isclose(expected, actual, rel_tol=1e-6):
    assert math.isclose(expected.real, actual.real, rel_tol=rel_tol), "Real parts not close: {} vs {}".format(expected.real, actual.real)
    assert math.isclose(expected.imag, actual.imag, rel_tol=rel_tol), "Imaginary parts not close: {} vs {}".format(expected.imag, actual.imag)

In [69]:
# Circuit parameters
v_volt = 8.5
r_ohm = 100
r2_ohm = 100 
r_on_switch = 1e-4 
r_off_switch = 1e8  
l_henry = 5
c_farad = 250e-6
deltaT = 1e-4
final_time = 1e-3
switch_on_time = 5*deltaT 

r_tot_switch_on = r_ohm*(r_on_switch + r2_ohm)/(r_ohm + r_on_switch + r2_ohm) # added for the test with switch, when the switch is on two resistors are in parallel

Analytical solution

In [70]:
eigenvalue0_expected = ((c_farad**2*r_ohm**2 - 4*l_henry*c_farad)**(1/2) - c_farad*r_ohm)/(2*c_farad*l_henry)
eigenvalue1_expected = -((c_farad**2*r_ohm**2 - 4*l_henry*c_farad)**(1/2) + c_farad*r_ohm)/(2*c_farad*l_henry)

eigenvalue0_switch_on_expected = ((c_farad**2*r_tot_switch_on**2 - 4*l_henry*c_farad)**(1/2) - c_farad*r_tot_switch_on)/(2*c_farad*l_henry) # added for the test with switch
eigenvale1_switch_on_expected = -((c_farad**2*r_tot_switch_on**2 - 4*l_henry*c_farad)**(1/2) + c_farad*r_tot_switch_on)/(2*c_farad*l_henry) # added for the test with switch

print('Expected eigenvalue 0: ' + str(eigenvalue0_expected))
print('Expected eigenvalue 1: ' + str(eigenvalue1_expected))

print('Expected eigenvalue 0 with r_tot_switch_on: ' + str(eigenvalue0_switch_on_expected)) # added for the test with switch
print('Expected eigenvalue 1 with r_tot_switch_on: ' + str(eigenvale1_switch_on_expected)) # added for the test with switch


Expected eigenvalue 0: (-9.999999999999998+26.45751311064591j)
Expected eigenvalue 1: (-10.000000000000002-26.45751311064591j)
Expected eigenvalue 0 with r_tot_switch_on: (-5.000002499998749+27.838821365136962j)
Expected eigenvalue 1 with r_tot_switch_on: (-5.000002499998752-27.838821365136962j)


Test 1: Eigenvalues extracted from simulation in DP domain match analytical solution. (NO SWITCH)

In [71]:
# DPsim DP simulation
name = 'DP_SinglePhaseRLC_Test'
log_dir = "logs/" + name
dpsimpy.Logger.set_log_dir(log_dir)

# Create nodes
gnd = dpsimpy.dp.SimNode.gnd
n1 = dpsimpy.dp.SimNode('n1')
n2 = dpsimpy.dp.SimNode('n2')
n3 = dpsimpy.dp.SimNode('n3')

# Create components
vs = dpsimpy.dp.ph1.VoltageSource('vs')
vs.set_parameters(V_ref=complex(v_volt,0))
r = dpsimpy.dp.ph1.Resistor('r')
r.set_parameters(R=r_ohm)
l = dpsimpy.dp.ph1.Inductor('l')
l.set_parameters(L=l_henry)
c = dpsimpy.dp.ph1.Capacitor('c')
c.set_parameters(C=c_farad)

# Set connections
vs.connect([gnd, n1])
l.connect([n1, n2])
c.connect([n2, n3])
r.connect([n3, gnd])

# Create topology
system = dpsimpy.SystemTopology(50, [n1, n2, n3], [vs, r, l, c])

# Configure and run simulation
sim = dpsimpy.Simulation(name)
sim.set_domain(dpsimpy.Domain.DP)
sim.set_system(system)
sim.do_eigenvalue_extraction(True)
sim.set_time_step(deltaT)
sim.set_final_time(final_time)
sim.run()

# Read log file
eigenvalues_timeseries = rt.read_timeseries_dpsim(log_dir + '/eigenvalues.csv')
eigenvalue0 = eigenvalues_timeseries['eigenvalues_0'].values
eigenvalue1 = eigenvalues_timeseries['eigenvalues_1'].values

# Assert
nSamples = int(final_time/deltaT)

assert len(eigenvalue0) == nSamples
assert len(eigenvalue1) == nSamples

for i in range(nSamples):
    assert_complex_isclose(eigenvalue0_expected, eigenvalue0[i], 1e-8)
    assert_complex_isclose(eigenvalue1_expected, eigenvalue1[i], 1e-8)

column number: 2
results length: 10
real column names: []
complex column names: ['eigenvalues_0', 'eigenvalues_1']


[02:41:25.378671 MnaSolverFactory info] creating KLUAdapter solver implementation


Test 2: Eigenvalues extracted from simulation in DP domain match analytical solution. (WITH SWITCH)

In [72]:
# DPsim DP simulation
name = 'DP_SinglePhaseRLC_Test'
log_dir = "logs/" + name
dpsimpy.Logger.set_log_dir(log_dir)

# Create nodes
gnd = dpsimpy.dp.SimNode.gnd
n1 = dpsimpy.dp.SimNode('n1')
n2 = dpsimpy.dp.SimNode('n2')
n3 = dpsimpy.dp.SimNode('n3')
n4 = dpsimpy.dp.SimNode('n4')

# Create components
vs = dpsimpy.dp.ph1.VoltageSource('vs')
vs.set_parameters(V_ref=complex(v_volt,0))
r = dpsimpy.dp.ph1.Resistor('r')
r.set_parameters(R=r_ohm)
l = dpsimpy.dp.ph1.Inductor('l')
l.set_parameters(L=l_henry)
c = dpsimpy.dp.ph1.Capacitor('c')
c.set_parameters(C=c_farad)

r2 = dpsimpy.dp.ph1.Resistor('r2') 
r2.set_parameters(R = r2_ohm) 
s1 = dpsimpy.dp.ph1.Switch('s1') 
s1.set_parameters(r_off_switch, r_on_switch, False) 


# Create switch events
s1Event = dpsimpy.event.SwitchEvent(switch_on_time, s1, True) 

# Set connections
vs.connect([gnd, n1])
l.connect([n1, n2])
c.connect([n2, n3])
r.connect([n3, gnd])
r2.connect([n4, gnd]) 
s1.connect([n3, n4]) 


# Create topology
system = dpsimpy.SystemTopology(50, [n1, n2, n3, n4], [vs, r, l, c, r2, s1]) 

# Configure and run simulation
sim = dpsimpy.Simulation(name)
sim.set_domain(dpsimpy.Domain.DP)
sim.set_system(system)
sim.add_event(s1Event) 
sim.do_eigenvalue_extraction(True)
sim.set_time_step(deltaT)
sim.set_final_time(final_time)
sim.run()

# Read log file
eigenvalues_timeseries = rt.read_timeseries_dpsim(log_dir + '/eigenvalues.csv')
eigenvalue0 = eigenvalues_timeseries['eigenvalues_0'].values
eigenvalue1 = eigenvalues_timeseries['eigenvalues_1'].values

# Assert
nSamples = int(final_time/deltaT)

assert len(eigenvalue0) == nSamples
assert len(eigenvalue1) == nSamples

for i in range(nSamples):
    if (i+1)*deltaT < switch_on_time :
        print((i+1)*deltaT)
        assert_complex_isclose(eigenvalue0_expected, eigenvalue0[i], 1e-5)  # A tolerance of 1e-6 or less results in an error
        assert_complex_isclose(eigenvalue1_expected, eigenvalue1[i], 1e-5)
    else:
        print((i+1)*deltaT)
        assert_complex_isclose(eigenvalue0_switch_on_expected, eigenvalue0[i], 1e-6)
        assert_complex_isclose(eigenvale1_switch_on_expected, eigenvalue1[i], 1e-6)
    

5.000000e-04: Handle event time
column number: 2
results length: 10
real column names: []
complex column names: ['eigenvalues_0', 'eigenvalues_1']
0.0001
0.0002
0.00030000000000000003
0.0004
0.0005
0.0006000000000000001
0.0007
0.0008
0.0009000000000000001
0.001


[02:41:25.475879 MnaSolverFactory info] creating KLUAdapter solver implementation


Test 3: Eigenvalues extracted from simulation in EMT domain match analytical solution. (NO SWITCH)

In [73]:
# DPsim EMT simulation
name = 'EMT_SinglePhaseRLC_Test'
log_dir = "logs/" + name
dpsimpy.Logger.set_log_dir(log_dir)

# Create nodes
gnd = dpsimpy.emt.SimNode.gnd
n1 = dpsimpy.emt.SimNode('n1')
n2 = dpsimpy.emt.SimNode('n2')
n3 = dpsimpy.emt.SimNode('n3')

# Create components
vs = dpsimpy.emt.ph1.VoltageSource('vs')
vs.set_parameters(V_ref=complex(v_volt,0))
r = dpsimpy.emt.ph1.Resistor('r')
r.set_parameters(R=r_ohm)
l = dpsimpy.emt.ph1.Inductor('l')
l.set_parameters(L=l_henry)
c = dpsimpy.emt.ph1.Capacitor('c')
c.set_parameters(C=c_farad)

# Set connections
vs.connect([gnd, n1])
l.connect([n1, n2])
c.connect([n2, n3])
r.connect([n3, gnd])

# Create topology
system = dpsimpy.SystemTopology(50, [n1, n2, n3], [vs, r, l, c])

# Configure and run simulation
sim = dpsimpy.Simulation(name)
sim.set_domain(dpsimpy.Domain.EMT)
sim.set_system(system)
sim.do_eigenvalue_extraction(True)
sim.set_time_step(deltaT)
sim.set_final_time(final_time)
sim.run()

# Read log file
eigenvalues_timeseries = rt.read_timeseries_dpsim(log_dir + '/eigenvalues.csv')
eigenvalue0 = eigenvalues_timeseries['eigenvalues_0'].values
eigenvalue1 = eigenvalues_timeseries['eigenvalues_1'].values

# Assert
nSamples = int(final_time/deltaT)

assert len(eigenvalue0) == nSamples
assert len(eigenvalue1) == nSamples

for i in range(nSamples):
    assert_complex_isclose(eigenvalue0_expected, eigenvalue0[i], 1e-8)
    assert_complex_isclose(eigenvalue1_expected, eigenvalue1[i], 1e-8)

column number: 2
results length: 10
real column names: []
complex column names: ['eigenvalues_0', 'eigenvalues_1']


[02:41:25.549053 MnaSolverFactory info] creating KLUAdapter solver implementation


Test 4: Eigenvalues extracted from simulation in EMT domain match analytical solution. (WITH SWITCH)

In [74]:
# DPsim EMT simulation
name = 'EMT_SinglePhaseRLC_Test'
log_dir = "logs/" + name
dpsimpy.Logger.set_log_dir(log_dir)

# Create nodes
gnd = dpsimpy.emt.SimNode.gnd
n1 = dpsimpy.emt.SimNode('n1')
n2 = dpsimpy.emt.SimNode('n2')
n3 = dpsimpy.emt.SimNode('n3')
n4 = dpsimpy.emt.SimNode('n4')

# Create components
vs = dpsimpy.emt.ph1.VoltageSource('vs')
vs.set_parameters(V_ref=complex(v_volt,0))
r = dpsimpy.emt.ph1.Resistor('r')
r.set_parameters(R=r_ohm)
l = dpsimpy.emt.ph1.Inductor('l')
l.set_parameters(L=l_henry)
c = dpsimpy.emt.ph1.Capacitor('c')
c.set_parameters(C=c_farad)

r2 = dpsimpy.emt.ph1.Resistor('r2') 
r2.set_parameters(R = r2_ohm) 
s1 = dpsimpy.emt.ph1.Switch('s1') 
s1.set_parameters(r_off_switch, r_on_switch, False) 


# Create switch events
s1Event = dpsimpy.event.SwitchEvent(switch_on_time, s1, True) 

# Set connections
vs.connect([gnd, n1])
l.connect([n1, n2])
c.connect([n2, n3])
r.connect([n3, gnd])
r2.connect([n4, gnd])
s1.connect([n3, n4]) 

# Create topology
system = dpsimpy.SystemTopology(50, [n1, n2, n3, n4], [vs, r, l, c, r2, s1]) 

# Configure and run simulation
sim = dpsimpy.Simulation(name)
sim.set_domain(dpsimpy.Domain.EMT)
sim.set_system(system)
sim.add_event(s1Event) 
sim.do_eigenvalue_extraction(True)
sim.set_time_step(deltaT)
sim.set_final_time(final_time)
sim.run()

# Read log file
eigenvalues_timeseries = rt.read_timeseries_dpsim(log_dir + '/eigenvalues.csv')
eigenvalue0 = eigenvalues_timeseries['eigenvalues_0'].values
eigenvalue1 = eigenvalues_timeseries['eigenvalues_1'].values

# Assert
nSamples = int(final_time/deltaT)

assert len(eigenvalue0) == nSamples
assert len(eigenvalue1) == nSamples

for i in range(nSamples):
    if (i+1)*deltaT < switch_on_time :
        print((i+1)*deltaT)
        assert_complex_isclose(eigenvalue0_expected, eigenvalue0[i], 1e-5) # A tolerance of 1e-6 or less results in an error
        assert_complex_isclose(eigenvalue1_expected, eigenvalue1[i], 1e-5)
    else:
        print((i+1)*deltaT)
        assert_complex_isclose(eigenvalue0_switch_on_expected, eigenvalue0[i], 1e-7)
        assert_complex_isclose(eigenvale1_switch_on_expected, eigenvalue1[i], 1e-7)
    

5.000000e-04: Handle event time
column number: 2
results length: 10
real column names: []
complex column names: ['eigenvalues_0', 'eigenvalues_1']
0.0001
0.0002
0.00030000000000000003
0.0004
0.0005
0.0006000000000000001
0.0007
0.0008
0.0009000000000000001
0.001


[02:41:25.628977 MnaSolverFactory info] creating KLUAdapter solver implementation
